# DiffEq and high precision floats 


- https://github.com/JuliaMath/DoubleFloats.jl
- https://github.com/dzhang314/MultiFloats.jl
- https://github.com/JuliaMath/Quadmath.jl
- https://github.com/JeffreySarnoff/ArbNumerics.jl


In [1]:
using DifferentialEquations, BenchmarkTools

In [2]:
using StaticArrays

In [3]:
using DoubleFloats  # Double16, Double32, Double64
using MultiFloats  # Float64x2, Float64x3, Float64x4, ...
using Quadmath  # Float128
using ArbNumerics  # ArbFloat, ArbReal, ArbComplex 

In [4]:
using Profile

In [89]:
using SimpleDiffEq

## Compute sqrt(2)

1.41421356237309504880168872420969807856967187537694807317667973799

In [30]:
sqrt(BigFloat(2.0))

1.414213562373095048801688724209698078569671875376948073176679737990732478462102

In [31]:
setprecision(ArbFloat, 250)
sqrt(ArbFloat(2.0))

1.41421356237309504880168872420969807856967187537694807317667973799073247846

In [32]:
sqrt(Float64x4(2.0))

1.414213562373095048801688724209698078569671875376948073176679738104

In [33]:
sqrt(Double64(2.0))

1.41421356237309504880168872420968161

In [34]:
sqrt(Float128(2.0))

1.41421356237309504880168872420969818e+00

## Solve ODEs (Lorenz equation) 

- Code optimization: https://docs.sciml.ai/DiffEqDocs/stable/tutorials/faster_ode_example/
- Solver recommendation: https://docs.sciml.ai/DiffEqDocs/stable/solvers/ode_solve/#Non-Stiff-Problems 

In [56]:
function lorenz(u, p, t)
    dx = 10.0 * (u[2] - u[1])
    dy = u[1] * (28.0 - u[3]) - u[2]
    dz = u[1] * u[2] - (8 / 3) * u[3]
    [dx, dy, dz]
end

function lorenz!(du, u, p, t)
    du[1] = 10.0 * (u[2] - u[1])
    du[2] = u[1] * (28.0 - u[3]) - u[2]
    du[3] = u[1] * u[2] - (8 / 3) * u[3]
    nothing
end

function lorenz_static(u, p, t)
    dx = 10.0 * (u[2] - u[1])
    dy = u[1] * (28.0 - u[3]) - u[2]
    dz = u[1] * u[2] - (8 / 3) * u[3]
    SA[dx, dy, dz]
end

lorenz_static (generic function with 1 method)

### 1. Float64

In [105]:
T = Float64;

tspan = T.((0.0, 100.0))
u0 = T.([1.0; 0.0; 0.0])
u0_static = SVector{3}(u0)

3-element SVector{3, Float64} with indices SOneTo(3):
 1.0
 0.0
 0.0

In [106]:
du = similar(u0);
p = [];
t = 0;

@btime lorenz(u0, p, t)
@btime lorenz!(du, u0, p, t)
@btime lorenz_static(u0_static, p, t)

  55.584 ns (1 allocation: 80 bytes)
  36.170 ns (0 allocations: 0 bytes)
  46.127 ns (1 allocation: 32 bytes)


3-element SVector{3, Float64} with indices SOneTo(3):
 -10.0
  28.0
   0.0

#### 1.1 out-of-place form 

In [107]:
prob = ODEProblem(lorenz, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  738.375 μs (25629 allocations: 1.96 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
   0.0
 100.0
u: 2-element Vector{Vector{Float64}}:
 [1.0, 0.0, 0.0]
 [-3.427697472901175, -2.710082989508757, 22.51738742249024]

#### 1.2 in-place form 

In [108]:
prob = ODEProblem(lorenz!, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  175.834 μs (43 allocations: 3.44 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
   0.0
 100.0
u: 2-element Vector{Vector{Float64}}:
 [1.0, 0.0, 0.0]
 [12.521672411415425, 10.65626263847737, 34.17691416212591]

#### 1.3 StaticArrays

In [109]:
prob = ODEProblem(lorenz_static, u0_static, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  90.833 μs (22 allocations: 2.14 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
   0.0
 100.0
u: 2-element Vector{SVector{3, Float64}}:
 [1.0, 0.0, 0.0]
 [12.521672411415425, 10.65626263847737, 34.17691416212591]

### 2. BigFloat

In [100]:
T = BigFloat;

tspan = T.((0.0, 100.0))
u0 = T.([1.0; 0.0; 0.0])
u0_static = SVector{3}(u0)

3-element SVector{3, BigFloat} with indices SOneTo(3):
 1.0
 0.0
 0.0

In [101]:
du = similar(u0);
p = [];
t = 0;

@btime lorenz(u0, p, t)
@btime lorenz!(du, u0, p, t)
@btime lorenz_static(u0_static, p, t)

  474.704 ns (17 allocations: 912 bytes)
  454.949 ns (16 allocations: 832 bytes)
  468.112 ns (17 allocations: 864 bytes)


3-element SVector{3, BigFloat} with indices SOneTo(3):
 -10.0
  28.0
   0.0

#### 2.1 out-of-place form 

In [102]:
prob = ODEProblem(lorenz, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  11.682 ms (377821 allocations: 19.42 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
   0.0
 100.0
u: 2-element Vector{Vector{BigFloat}}:
 [1.0, 0.0, 0.0]
 [-12.99282616832083724111929082294932127785542212696109239677389582243509342377402, -17.12834735011618008720611715130716507235635464337586942842731062225327697827589, 27.98178978214227320948792375356788773638545615107835580536526232990719820556955]

#### 2.2 in-place form 

In [103]:
prob = ODEProblem(lorenz!, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  11.144 ms (368257 allocations: 18.26 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
   0.0
 100.0
u: 2-element Vector{Vector{BigFloat}}:
 [1.0, 0.0, 0.0]
 [-12.99282616832083724111929082294932127785542212696109239677389582243509342377402, -17.12834735011618008720611715130716507235635464337586942842731062225327697827589, 27.98178978214227320948792375356788773638545615107835580536526232990719820556955]

#### 2.3 StaticArrays

In [104]:
prob = ODEProblem(lorenz_static, u0_static, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  10.927 ms (361806 allocations: 17.94 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
   0.0
 100.0
u: 2-element Vector{SVector{3, BigFloat}}:
 [1.0, 0.0, 0.0]
 [-12.99282616832083724111929082294932127785542212696109239677389582243509342377402, -17.12834735011618008720611715130716507235635464337586942842731062225327697827589, 27.98178978214227320948792375356788773638545615107835580536526232990719820556955]

### 3. ArbFloat

In [95]:
T = ArbFloat;

tspan = T.((0.0, 100.0))
u0 = T.([1.0; 0.0; 0.0])
u0_static = SVector{3}(u0)

3-element SVector{3, ArbFloat{274}} with indices SOneTo(3):
 1.0
 0
 0

In [96]:
du = similar(u0);
p = [];
t = 0;

@btime lorenz(u0, p, t)
@btime lorenz!(du, u0, p, t)
@btime lorenz_static(u0_static, p, t)

  725.000 ns (50 allocations: 1.19 KiB)
  697.846 ns (49 allocations: 1.11 KiB)
  698.129 ns (50 allocations: 1.14 KiB)


3-element SVector{3, ArbFloat{274}} with indices SOneTo(3):
 -10.0
  28.0
   0

#### 3.1 out-of-place form 

In [97]:
prob = ODEProblem(lorenz, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  19.483 ms (1098180 allocations: 29.42 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
   0
 100.0
u: 2-element Vector{Vector{ArbFloat{274}}}:
 [1.0, 0, 0]
 [-12.9928261683208372411192908229493212778554289441605806883431696610457974815, -17.1283473501161800872061171513071650723564160668219176367233667884382821946, 27.9817897821422732094879237535678877363853933250455681765665031700953449333]

#### 3.2 in-place form 

In [98]:
prob = ODEProblem(lorenz!, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  17.783 ms (1018232 allocations: 25.55 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
   0
 100.0
u: 2-element Vector{Vector{ArbFloat{274}}}:
 [1.0, 0, 0]
 [-12.9928261683208372411192908229493212778554289440925136735880513737341555929, -17.1283473501161800872061171513071650723564160662086290545637493351666104757, 27.9817897821422732094879237535678877363853933256728610272606466576800795083]

#### 3.3 StaticArrays

In [99]:
prob = ODEProblem(lorenz_static, u0_static, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  17.058 ms (976571 allocations: 24.36 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
   0
 100.0
u: 2-element Vector{SVector{3, ArbFloat{274}}}:
 [1.0, 0, 0]
 [-12.9928261683208372411192908229493212778554289440925136735880513737341555929, -17.1283473501161800872061171513071650723564160662086290545637493351666104757, 27.9817897821422732094879237535678877363853933256728610272606466576800795083]

### 4. Float64x4

In [80]:
T = Float64x4;

tspan = T.((0.0, 100.0))
u0 = T.([1.0; 0.0; 0.0])
u0_static = SVector{3}(u0)

3-element SVector{3, MultiFloat{Float64, 4}} with indices SOneTo(3):
 1.0
 0.0
 0.0

In [81]:
du = similar(u0);
p = [];
t = 0;

@btime lorenz(u0, p, t)
@btime lorenz!(du, u0, p, t)
@btime lorenz_static(u0_static, p, t)

  127.973 ns (1 allocation: 160 bytes)
  109.397 ns (0 allocations: 0 bytes)
  118.830 ns (1 allocation: 112 bytes)


3-element SVector{3, MultiFloat{Float64, 4}} with indices SOneTo(3):
 -10.0
  28.0
   0.0

#### 4.1 out-of-place form 

In [92]:
prob = ODEProblem(lorenz, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  3.208 ms (25629 allocations: 3.91 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
   0.0
 100.0
u: 2-element Vector{Vector{MultiFloat{Float64, 4}}}:
 [1.0, 0.0, 0.0]
 [-12.992826168320837241119290822953483197183271045260183045759012041318, -17.128347350116180087206117151344664257722747394911352815944435975485, 27.98178978214227320948792375352953226789726176880837557973897345673]

#### 4.2 in-place form 

In [93]:
prob = ODEProblem(lorenz!, u0, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  3.469 ms (43 allocations: 5.47 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
   0.0
 100.0
u: 2-element Vector{Vector{MultiFloat{Float64, 4}}}:
 [1.0, 0.0, 0.0]
 [-12.992826168320837241119290822953483197183271045260183045759012041318, -17.128347350116180087206117151344664257722747394911352815944435975485, 27.98178978214227320948792375352953226789726176880837557973897345673]

#### 4.3 StaticArrays

In [94]:
prob = ODEProblem(lorenz_static, u0_static, tspan)
@btime sol = solve(prob, RK4(), adaptive=false, dt=2^(-4), save_everystep=false)

  2.783 ms (22 allocations: 4.61 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
   0.0
 100.0
u: 2-element Vector{SVector{3, MultiFloat{Float64, 4}}}:
 [1.0, 0.0, 0.0]
 [-12.992826168320837241119290822953483197183271045260183045759012041318, -17.128347350116180087206117151344664257722747394911352815944435975485, 27.98178978214227320948792375352953226789726176880837557973897345673]

## Solve ODEs (FPU problem)

In [5]:
include("./setups/fpu.jl");

In [45]:
# function odedef(u,p,t)
#     halfomegasquared = 0.5 * p^2
#     du = zero(u)
#     du[1] = halfomegasquared * (u[8] - u[7]) - 4 * u[7].^3
#     du[2] = - halfomegasquared * (u[8] - u[7]) + 4 * (u[9] - u[8]).^3
#     du[3] = halfomegasquared * (u[10] - u[9]) - 4 * (u[9] - u[8]).^3 
#     du[4] = - halfomegasquared * (u[10] - u[9]) + 4 * (u[11] - u[10]).^3
#     du[5] = halfomegasquared * (u[12] - u[11]) - 4 * (u[11] - u[10]).^3 
#     du[6] = - halfomegasquared * (u[12] - u[11]) + 4 * (- u[12]).^3
#     du[7:end] = @view u[1:6]
#     du 
# end

# function odedef!(du,u,p,t)
#     halfomegasquared = 0.5 * p^2
#     du[1] = halfomegasquared * (u[8] - u[7]) - 4 * u[7].^3
#     du[2] = - halfomegasquared * (u[8] - u[7]) + 4 * (u[9] - u[8]).^3
#     du[3] = halfomegasquared * (u[10] - u[9]) - 4 * (u[9] - u[8]).^3 
#     du[4] = - halfomegasquared * (u[10] - u[9]) + 4 * (u[11] - u[10]).^3
#     du[5] = halfomegasquared * (u[12] - u[11]) - 4 * (u[11] - u[10]).^3 
#     du[6] = - halfomegasquared * (u[12] - u[11]) + 4 * (- u[12]).^3
#     du[7:end] = @view u[1:6]
#     nothing
# end

# function odedef_static(u,p,t)
# #     du = similar(u)
# #     du[1] = halfomegasquared * (u[8] - u[7]) - 4 * u[7].^3
# #     du[2] = - halfomegasquared * (u[8] - u[7]) + 4 * (u[9] - u[8]).^3
# #     du[3] = halfomegasquared * (u[10] - u[9]) - 4 * (u[9] - u[8]).^3 
# #     du[4] = - halfomegasquared * (u[10] - u[9]) + 4 * (u[11] - u[10]).^3
# #     du[5] = halfomegasquared * (u[12] - u[11]) - 4 * (u[11] - u[10]).^3 
# #     du[6] = - halfomegasquared * (u[12] - u[11]) + 4 * (- u[12]).^3
# #     du[7:end] = @view u[1:6]
# #     du 
#     halfomegasquared = 0.5 * p^2
#     du1 = halfomegasquared * (u[8] - u[7]) - 4 * u[7].^3
#     du2 = - halfomegasquared * (u[8] - u[7]) + 4 * (u[9] - u[8]).^3
#     du3 = halfomegasquared * (u[10] - u[9]) - 4 * (u[9] - u[8]).^3 
#     du4 = - halfomegasquared * (u[10] - u[9]) + 4 * (u[11] - u[10]).^3
#     du5 = halfomegasquared * (u[12] - u[11]) - 4 * (u[11] - u[10]).^3 
#     du6 = - halfomegasquared * (u[12] - u[11]) + 4 * (- u[12]).^3
# #     du[7:end] = @view u[1:6]
#     SA[du1, du2, du3, du4, du5, du6, (@view u[1:6])...]
# end

In [6]:
function A(du, u, p, t) 
    """Compute ddu, second order time derivative of u"""
    
    halfomegasquared = p
    
    ddu = zero(u)
    ddu[1] = halfomegasquared * (u[2] - u[1]) - 4 * u[1].^3
    ddu[2] = - halfomegasquared * (u[2] - u[1]) + 4 * (u[3] - u[2]).^3
    ddu[3] = halfomegasquared * (u[4] - u[3]) - 4 * (u[3] - u[2]).^3 
    ddu[4] = - halfomegasquared * (u[4] - u[3]) + 4 * (u[5] - u[4]).^3
    ddu[5] = halfomegasquared * (u[6] - u[5]) - 4 * (u[5] - u[4]).^3 
    ddu[6] = - halfomegasquared * (u[6] - u[5]) + 4 * (- u[6]).^3
    
    ddu
end

function A!(ddu, du, u, p, t) 
    """Compute ddu, second order time derivative of u"""
    
    halfomegasquared = p
    
#     u_odd = @view u[1:2:end]
#     u_even = @view u[2:2:end]
#     ddu_odd = @view ddu[1:2:end]
#     ddu_even = @view ddu[2:2:end]
    
#     @. ddu_odd = halfomegasquared * (u_even .- u_odd)
#     @. ddu_even = - ddu_odd
    
#     u_odd = @view u[3:2:end]
#     u_even = @view u[2:2:end-1]
#     ddu_odd = @view ddu[3:2:end]
#     ddu_even = @view ddu[2:2:end-1]
    
#     ddu[1] -= 4 * u[1].^3 
#     @. ddu_odd -= 4 * (u_odd .- u_even).^3 
#     @. ddu_even += 4 * (u_odd .- u_even).^3 
#     ddu[end] += 4 * (-u[end]).^3 
    
    ddu[1] = halfomegasquared * (u[2] - u[1]) - 4 * u[1].^3
    ddu[2] = - halfomegasquared * (u[2] - u[1]) + 4 * (u[3] - u[2]).^3
    ddu[3] = halfomegasquared * (u[4] - u[3]) - 4 * (u[3] - u[2]).^3 
    ddu[4] = - halfomegasquared * (u[4] - u[3]) + 4 * (u[5] - u[4]).^3
    ddu[5] = halfomegasquared * (u[6] - u[5]) - 4 * (u[5] - u[4]).^3 
    ddu[6] = - halfomegasquared * (u[6] - u[5]) + 4 * (- u[6]).^3
    
    nothing 
end

function A_static(du, u, p, t) 
    """Compute ddu, second order time derivative of u"""
    
    halfomegasquared = p
    
    ddu1 = halfomegasquared * (u[2] - u[1]) - 4 * u[1].^3
    ddu2 = - halfomegasquared * (u[2] - u[1]) + 4 * (u[3] - u[2]).^3
    ddu3 = halfomegasquared * (u[4] - u[3]) - 4 * (u[3] - u[2]).^3 
    ddu4 = - halfomegasquared * (u[4] - u[3]) + 4 * (u[5] - u[4]).^3
    ddu5 = halfomegasquared * (u[6] - u[5]) - 4 * (u[5] - u[4]).^3 
    ddu6 = - halfomegasquared * (u[6] - u[5]) + 4 * (- u[6]).^3
    
    SA[ddu1, ddu2, ddu3, ddu4, ddu5, ddu6]
end

A_static (generic function with 1 method)

### 1. Float64

In [35]:
T = Float64;

tspan = T.([0.0, 1.0])
p0, q0 = initial_condition(omega=T(300.))

([0.0, 1.4142135623730951, 0.0, 0.0, 0.0, 0.0], [0.7047497585825924, 0.7094638037905027, 0.0, 0.0, 0.0, 0.0])

In [36]:
ddq = similar(q0)
p0_static = SVector{6}(p0)
q0_static = SVector{6}(q0)
p = 0.5 * T(300.)^2;
t = 0;

@btime A(p0, q0, p, t)
@btime A!(ddq, p0, q0, p, t)
@btime A_static(p0_static, q0_static, p, t)

  51.935 ns (1 allocation: 112 bytes)
  30.517 ns (0 allocations: 0 bytes)
  33.743 ns (1 allocation: 64 bytes)


6-element SVector{6, Float64} with indices SOneTo(6):
  210.73191584114034
 -213.56043724679068
    1.428402890827185
    0.0
    0.0
   -0.0

#### 1.1 out-of-place form 

In [37]:
prob = SecondOrderODEProblem(A, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  38.997 ms (1441887 allocations: 154.01 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{Float64, Tuple{Vector{Float64}, Vector{Float64}}}}:
 ([0.0, 1.4142135623730951, 0.0, 0.0, 0.0, 0.0], [0.7047497585825924, 0.7094638037905027, 0.0, 0.0, 0.0, 0.0])
 ([-1.45916982940642, -0.062406859344893484, 0.560587522602886, 0.5712123500342404, 0.019897277818265332, 0.01988464657572776], [0.5311236989295888, 0.5263588641879837, 0.3881409024418263, 0.38815693764835024, 0.0028011438887681696, 0.002798612924126341])

#### 1.2 in-place form 

In [38]:
prob = SecondOrderODEProblem(A!, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  8.188 ms (94 allocations: 8.44 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{Float64, Tuple{Vector{Float64}, Vector{Float64}}}}:
 ([0.0, 1.4142135623730951, 0.0, 0.0, 0.0, 0.0], [0.7047497585825924, 0.7094638037905027, 0.0, 0.0, 0.0, 0.0])
 ([-1.4591698294042295, -0.06240685934705376, 0.5605875226027222, 0.571212350034375, 0.019897277818264815, 0.01988464657572904], [0.5311236989296018, 0.5263588641880056, 0.38814090244182625, 0.38815693764834874, 0.0028011438887682095, 0.0027986129241263946])

#### 1.3 StaticArrays

In [39]:
prob = SecondOrderODEProblem(A_static, p0_static, q0_static, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  5.573 ms (22 allocations: 4.50 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{Float64}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{Float64, Tuple{SVector{6, Float64}, SVector{6, Float64}}}}:
 ([0.0, 1.4142135623730951, 0.0, 0.0, 0.0, 0.0], [0.7047497585825924, 0.7094638037905027, 0.0, 0.0, 0.0, 0.0])
 ([-1.4591698294042295, -0.06240685934705376, 0.5605875226027222, 0.571212350034375, 0.019897277818264815, 0.01988464657572904], [0.5311236989296018, 0.5263588641880056, 0.38814090244182625, 0.38815693764834874, 0.0028011438887682095, 0.0027986129241263946])

solutions don't match?

### 2. BigFloat

In [40]:
T = BigFloat;

tspan = T.([0.0, 1.0])
p0, q0 = initial_condition(omega=T(300.))

(BigFloat[0.0, 1.414213562373095048801688724209698078569671875376948073176679737990732478462102, 0.0, 0.0, 0.0, 0.0], BigFloat[0.7047497585825923659861748808978328758205531512295124564663787360987150184336224, 0.7094638037905026828155138433118652027491187241474356167103010018920174600284883, 0.0, 0.0, 0.0, 0.0])

In [41]:
ddq = similar(q0)
p0_static = SVector{6}(p0)
q0_static = SVector{6}(q0)
p = 0.5 * T(300.)^2;
t = 0;

@btime A(p0, q0, p, t)
@btime A!(ddq, p0, q0, p, t)
@btime A_static(p0_static, q0_static, p, t)

  1.796 μs (84 allocations: 4.32 KiB)
  1.738 μs (79 allocations: 4.02 KiB)
  1.717 μs (80 allocations: 4.09 KiB)


6-element SVector{6, BigFloat} with indices SOneTo(6):
  210.7319158411410942066637526197403305203122641449184411293786112455885112942321
 -213.5604372467914425106037168474080073239901792071306957389368491668858422999887
    1.428402890827185190350408215953295538539397900588484762434888468275970531023124
    0.0
    0.0
   -0.0

#### 2.1 out-of-place form 

In [42]:
prob = SecondOrderODEProblem(A, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  1.399 s (44040826 allocations: 2.20 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{BigFloat, Tuple{Vector{BigFloat}, Vector{BigFloat}}}}:
 (BigFloat[0.0, 1.414213562373095048801688724209698078569671875376948073176679737990732478462102, 0.0, 0.0, 0.0, 0.0], BigFloat[0.7047497585825923659861748808978328758205531512295124564663787360987150184336224, 0.7094638037905026828155138433118652027491187241474356167103010018920174600284883, 0.0, 0.0, 0.0, 0.0])
 (BigFloat[-1.459169829409809452092738087268239497075617757685958678346533298538987402830431, -0.06240685934150673726030525655232602073420964438329327549903410754236298058741796, 0.5605875226019461877557092850055221480354608127195363498994491916332333404294306, 0.5712123500351554596348396011246948929639436548874871246805443328011522082265264, 0.01989727781826256040113088132079923164405284091451109823365875978648488877725824, 0.01988464657573250479800289669972856851608780808545328021839670617322457329

#### 2.2 in-place form 

In [43]:
prob = SecondOrderODEProblem(A!, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  1.532 s (47153806 allocations: 2.29 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{BigFloat, Tuple{Vector{BigFloat}, Vector{BigFloat}}}}:
 (BigFloat[0.0, 1.414213562373095048801688724209698078569671875376948073176679737990732478462102, 0.0, 0.0, 0.0, 0.0], BigFloat[0.7047497585825923659861748808978328758205531512295124564663787360987150184336224, 0.7094638037905026828155138433118652027491187241474356167103010018920174600284883, 0.0, 0.0, 0.0, 0.0])
 (BigFloat[-1.459169829409809452092738087268239497075617757685958678346533298538987402830431, -0.06240685934150673726030525655232602073420964438329327549903410754236298058741796, 0.5605875226019461877557092850055221480354608127195363498994491916332333404294306, 0.5712123500351554596348396011246948929639436548874871246805443328011522082265264, 0.01989727781826256040113088132079923164405284091451109823365875978648488877725824, 0.01988464657573250479800289669972856851608780808545328021839670617322457329

#### 2.3 StaticArrays

In [44]:
prob = SecondOrderODEProblem(A_static, p0_static, q0_static, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  1.362 s (41419352 allocations: 2.01 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{BigFloat}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{BigFloat, Tuple{SVector{6, BigFloat}, SVector{6, BigFloat}}}}:
 (BigFloat[0.0, 1.414213562373095048801688724209698078569671875376948073176679737990732478462102, 0.0, 0.0, 0.0, 0.0], BigFloat[0.7047497585825923659861748808978328758205531512295124564663787360987150184336224, 0.7094638037905026828155138433118652027491187241474356167103010018920174600284883, 0.0, 0.0, 0.0, 0.0])
 (BigFloat[-1.459169829409809452092738087268239497075617757685958678346533298538987402830431, -0.06240685934150673726030525655232602073420964438329327549903410754236298058741796, 0.5605875226019461877557092850055221480354608127195363498994491916332333404294306, 0.5712123500351554596348396011246948929639436548874871246805443328011522082265264, 0.01989727781826256040113088132079923164405284091451109823365875978648488877725824, 0.019884646575732504798002896699728568516087808085453280218396706173

### 3. ArbFloat

In [45]:
T = ArbFloat;

tspan = T.([0.0, 1.0])
p0, q0 = initial_condition(omega=T(300.))

(ArbFloat{274}[0, 1.41421356237309504880168872420969807856967187537694807317667973799073247846, 0, 0, 0, 0], ArbFloat{274}[0.704749758582592365986174880897832875820553151229512456466378736098715018434, 0.709463803790502682815513843311865202749118724147435616710301001892017460028, 0, 0, 0, 0])

In [46]:
ddq = similar(q0)
p0_static = SVector{6}(p0)
q0_static = SVector{6}(q0)
p = 0.5 * T(300.)^2;
t = 0;

@btime A(p0, q0, p, t)
@btime A!(ddq, p0, q0, p, t)
@btime A_static(p0_static, q0_static, p, t)

  6.192 μs (395 allocations: 10.88 KiB)
  6.108 μs (390 allocations: 10.66 KiB)
  6.083 μs (391 allocations: 10.72 KiB)


6-element SVector{6, ArbFloat{274}} with indices SOneTo(6):
  210.731915841141094206663752619740330520312264144918441129378611245588511295
 -213.5604372467914425106037168474080073239901792071306957389368491668858423
    1.42840289082718519035040821595329553853939790058848476243488846827597053102
    0
    0
    0

#### 3.1 out-of-place form 

In [47]:
prob = SecondOrderODEProblem(A, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  6.432 s (206146098 allocations: 5.33 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
 0
 1.0
u: 2-element Vector{ArrayPartition{ArbFloat{274}, Tuple{Vector{ArbFloat{274}}, Vector{ArbFloat{274}}}}}:
 (ArbFloat{274}[0, 1.41421356237309504880168872420969807856967187537694807317667973799073247846, 0, 0, 0, 0], ArbFloat{274}[0.704749758582592365986174880897832875820553151229512456466378736098715018434, 0.709463803790502682815513843311865202749118724147435616710301001892017460028, 0, 0, 0, 0])
 (ArbFloat{274}[-1.45916982940980945209273808726823949707561775768595867834653329853898740276, -0.0624068593415067372603052565523260207342096443832932754990341075423629806564, 0.560587522601946187755709285005522148035460812719536349899449191633233340431, 0.571212350035155459634839601124694892963943654887487124680544332801152208223, 0.0198972778182625604011308813207992316440528409145110982336587597864848887771, 0.0198846465757325047980028966997285685160878080854532802183967061732245732918], ArbFloat{274

#### 3.2 in-place form 

In [48]:
prob = SecondOrderODEProblem(A!, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  6.235 s (205568412 allocations: 5.14 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
 0
 1.0
u: 2-element Vector{ArrayPartition{ArbFloat{274}, Tuple{Vector{ArbFloat{274}}, Vector{ArbFloat{274}}}}}:
 (ArbFloat{274}[0, 1.41421356237309504880168872420969807856967187537694807317667973799073247846, 0, 0, 0, 0], ArbFloat{274}[0.704749758582592365986174880897832875820553151229512456466378736098715018434, 0.709463803790502682815513843311865202749118724147435616710301001892017460028, 0, 0, 0, 0])
 (ArbFloat{274}[-1.45916982940980945209273808726823949707561775768595867834653329853898740276, -0.0624068593415067372603052565523260207342096443832932754990341075423629806564, 0.560587522601946187755709285005522148035460812719536349899449191633233340431, 0.571212350035155459634839601124694892963943654887487124680544332801152208223, 0.0198972778182625604011308813207992316440528409145110982336587597864848887771, 0.0198846465757325047980028966997285685160878080854532802183967061732245732918], ArbFloat{274

#### 3.3 StaticArrays

In [49]:
prob = SecondOrderODEProblem(A_static, p0_static, q0_static, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  5.546 s (185146852 allocations: 4.66 GiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{ArbFloat{274}}:
 0
 1.0
u: 2-element Vector{ArrayPartition{ArbFloat{274}, Tuple{SVector{6, ArbFloat{274}}, SVector{6, ArbFloat{274}}}}}:
 (ArbFloat{274}[0, 1.41421356237309504880168872420969807856967187537694807317667973799073247846, 0, 0, 0, 0], ArbFloat{274}[0.704749758582592365986174880897832875820553151229512456466378736098715018434, 0.709463803790502682815513843311865202749118724147435616710301001892017460028, 0, 0, 0, 0])
 (ArbFloat{274}[-1.45916982940980945209273808726823949707561775768595867834653329853898740276, -0.0624068593415067372603052565523260207342096443832932754990341075423629806564, 0.560587522601946187755709285005522148035460812719536349899449191633233340431, 0.571212350035155459634839601124694892963943654887487124680544332801152208223, 0.0198972778182625604011308813207992316440528409145110982336587597864848887771, 0.0198846465757325047980028966997285685160878080854532802183967061732245732918], ArbF

### 4. Float64x4

In [50]:
Base.round(x::MultiFloat{Float64, 4}, y::RoundingMode) = MultiFloat{Float64, 4}(Base.round(Float64(x),y))
Base.trunc(x::Type{Int64}, y::MultiFloat{Float64, 4}) = Base.trunc(x::Type{Int64}, Float64(y))

In [110]:
T = Float64x4;

tspan = T.([0.0, 1.0])
p0, q0 = initial_condition(omega=T(300.))

(MultiFloat{Float64, 4}[0.0, 1.414213562373095048801688724209698078569671875376948073176679738104, 0.0, 0.0, 0.0, 0.0], MultiFloat{Float64, 4}[0.70474975858259236598617488089783287582055315122951245646637873603984, 0.70946380379050268281551384331186520274911872414743561671030100179855, 0.0, 0.0, 0.0, 0.0])

In [111]:
ddq = similar(q0)
p0_static = SVector{6}(p0)
q0_static = SVector{6}(q0)
p = 0.5 * T(300.)^2;
t = 0;

@btime A(p0, q0, p, t)
@btime A!(ddq, p0, q0, p, t)
@btime A_static(p0_static, q0_static, p, t)

  659.638 ns (1 allocation: 256 bytes)
  630.147 ns (0 allocations: 0 bytes)
  592.877 ns (1 allocation: 208 bytes)


6-element SVector{6, MultiFloat{Float64, 4}} with indices SOneTo(6):
  210.73191584114109420666375261974033052031226414491844112937860969
 -213.56043724679144251060371684740800732399017920713069573893684761017
    1.428402890827185190350408215953295538539397900588484762434888467702
    0.0
    0.0
    0.0

#### 4.1 out-of-place form 

In [115]:
prob = SecondOrderODEProblem(A, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  336.772 ms (1442119 allocations: 352.02 MiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{MultiFloat{Float64, 4}, Tuple{Vector{MultiFloat{Float64, 4}}, Vector{MultiFloat{Float64, 4}}}}}:
 (MultiFloat{Float64, 4}[0.0, 1.414213562373095048801688724209698078569671875376948073176679738104, 0.0, 0.0, 0.0, 0.0], MultiFloat{Float64, 4}[0.70474975858259236598617488089783287582055315122951245646637873603984, 0.70946380379050268281551384331186520274911872414743561671030100179855, 0.0, 0.0, 0.0, 0.0])
 (MultiFloat{Float64, 4}[-1.4591698294098094520927380872682394970756177576859586783465331332825, -0.0624068593415067372603052565523260207342096443832932754990342748145, 0.5605875226019461877557092850055221480354608127195363498994492407018, 0.571212350035155459634839601124694892963943654887487124680544285700491, 0.019897277818262560401130881320799231644052840914511098233658759793911, 0.0198846465757325047980028966997285685160878080854532802183967061712

#### 4.2 in-place form 

In [120]:
prob = SecondOrderODEProblem(A!, p0, q0, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  331.768 ms (329 allocations: 29.30 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{MultiFloat{Float64, 4}, Tuple{Vector{MultiFloat{Float64, 4}}, Vector{MultiFloat{Float64, 4}}}}}:
 (MultiFloat{Float64, 4}[0.0, 1.414213562373095048801688724209698078569671875376948073176679738104, 0.0, 0.0, 0.0, 0.0], MultiFloat{Float64, 4}[0.70474975858259236598617488089783287582055315122951245646637873603984, 0.70946380379050268281551384331186520274911872414743561671030100179855, 0.0, 0.0, 0.0, 0.0])
 (MultiFloat{Float64, 4}[-1.4591698294098094520927380872682394970756177576859586783465331332825, -0.0624068593415067372603052565523260207342096443832932754990342748145, 0.5605875226019461877557092850055221480354608127195363498994492407018, 0.571212350035155459634839601124694892963943654887487124680544285700491, 0.019897277818262560401130881320799231644052840914511098233658759793911, 0.0198846465757325047980028966997285685160878080854532802183967061712

#### 4.3 StaticArrays

In [117]:
prob = SecondOrderODEProblem(A_static, p0_static, q0_static, tspan, p)
@btime sol = solve(prob, KahanLi8(), adaptive=false, dt=2^(-14), save_everystep=false)

  270.243 ms (269 allocations: 23.80 KiB)


retcode: Success
Interpolation: 1st order linear
t: 2-element Vector{MultiFloat{Float64, 4}}:
 0.0
 1.0
u: 2-element Vector{ArrayPartition{MultiFloat{Float64, 4}, Tuple{SVector{6, MultiFloat{Float64, 4}}, SVector{6, MultiFloat{Float64, 4}}}}}:
 (MultiFloat{Float64, 4}[0.0, 1.414213562373095048801688724209698078569671875376948073176679738104, 0.0, 0.0, 0.0, 0.0], MultiFloat{Float64, 4}[0.70474975858259236598617488089783287582055315122951245646637873603984, 0.70946380379050268281551384331186520274911872414743561671030100179855, 0.0, 0.0, 0.0, 0.0])
 (MultiFloat{Float64, 4}[-1.4591698294098094520927380872682394970756177576859586783465331332825, -0.0624068593415067372603052565523260207342096443832932754990342748145, 0.5605875226019461877557092850055221480354608127195363498994492407018, 0.571212350035155459634839601124694892963943654887487124680544285700491, 0.019897277818262560401130881320799231644052840914511098233658759793911, 0.01988464657573250479800289669972856851608780808545328021839

### Summary

1. For any dtype, different function forms (out-of-place, in-place, SA) produce same results. Float64 is an exception, where Float64 (out-of-place) != Float64 (in-place) = Float64 (SA). 

2. Unlike BigFloat and ArbFloat, Float64x4 avoids dynamic memory allocation. See performance for 1 evaluation of A(du, u, p, t), A!(ddu, du, u, p, t) and A_static(du, u, p, t). 
|       | Time | Memory allocations     |
| :---        |    :----:   |          :---: |
| Float64 (out-of-place)     | 51.935 ns | 1 allocation: 112 bytes | 
| Float64 (in-place)   | 30.517 ns  | 0 allocations: 0 bytes  |
| Float64 (SA)   | 33.743 ns  | 1 allocation: 64 bytes    |
| BigFloat (out-of-place)     | 1.796 μs | 84 allocations: 4.32 KiB |
| BigFloat (in-place)   | 1.738 μs  | 79 allocations: 4.02 KiB    |
| BigFloat (SA)   | 1.717 μs | 80 allocations: 4.09 KiB  |
| ArbFloat (out-of-place)     | 6.192 μs | 395 allocations: 10.88 KiB |
| ArbFloat (in-place)   |  6.108 μs  | 390 allocations: 10.66 KiB     |
| ArbFloat (SA)   | 6.083 μs   | 391 allocations: 10.72 KiB    |
| Float64x4 (out-of-place)     |661.658 ns | 1 allocation: 256 bytes |
| Float64x4 (in-place)   |   629.900 ns    |  0 allocations: 0 bytes  |
| Float64x4 (SA)   | 592.877 ns |   1 allocation: 208 bytes |

3. Overall performance: tspan=(0.0, 1.0), fine method = KahanLi8, fine stepsize = 2^(-14). As a result of 2 (also, with appropriate integration method), memory allocations of Float64x4 (in-place) and Float64x4 (SA) do not scale with problem size (number of fine steps).

|       | Time | Memory allocations     |
| :---        |    :----:   |          :---: |
| Float64 (out-of-place)     | 38.997 ms  | 1441887 allocations: 154.01 MiB         | 
| Float64 (in-place)   | 8.188 ms | 94 allocations: 8.44 KiB   |
| Float64 (SA)   | 5.573 ms | 22 allocations: 4.50 KiB      |
| BigFloat (out-of-place)     | 1.399 s   | 44040826 allocations: 2.20 GiB |
| BigFloat (in-place)   | 1.532 s   | 47153806 allocations: 2.29 GiB    |
| BigFloat (SA)   | 1.362 s |  41419352 allocations: 2.01 GiB     |
| ArbFloat (out-of-place)     | 6.432 s   | 206146098 allocations: 5.33 GiB  |
| ArbFloat (in-place)   | 6.235 s   | 205568412 allocations: 5.14 GiB     |
| ArbFloat (SA)   | 5.546 s   | 185146852 allocations: 4.66 GiB     |
| Float64x4 (out-of-place)     | 331.340 ms     | 1442119 allocations: 352.02 MiB  |
| Float64x4 (in-place)   | 332.328 ms    | 329 allocations: 29.30 KiB    |
| Float64x4 (SA)   | 270.356 ms  | 269 allocations: 23.80 KiB |


p1 at t=1.0: 
-1.4591698294 0980945209 2738087268 2394970756 1775768595 8678346533 298538987402830431  # BigFloat
-1.4591698294 0980945209 2738087268 2394970756 1775768595 8678346533 29853898740276  # ArbFloat
-1.4591698294 0980945209 2738087268 2394970756 1775768595 8678346533 1332825  # Float64x4

q6 at t=1.0: 
0.002798612924126472160119728865087628849994004087873201328794938604029738268913848  # BigFloat
0.0027986129241264721601197288650876288499940040878732013287949386040297382689  # ArbFloat
0.0027986129241264721601197288650876288499940040878732013287949386048415  # Float64x4